In [53]:
import sys 
import os
sys.path.append('..')

In [54]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt

In [55]:
appl_train = pd.read_csv('../data/dseb63_application_train.csv', index_col=0)

In [56]:
bureau = pd.read_csv('../data/dseb63_bureau_general_v2.csv', index_col=0)
bureau_columns = bureau.columns

new_bureau_columns = {col: 'BUREAU_' + col for col in bureau_columns if col != 'SK_ID_CURR'} 
bureau.rename(columns=new_bureau_columns, inplace=True)

In [57]:
installments = pd.read_csv('../data/dseb63_installment_gb.csv', index_col=0)
installments_columns = installments.columns

installments.drop(columns= [col for col in installments_columns if 'TARGET' in col], inplace=True)

new_installments_columns = {col: 'INSTALLMENTS_' + col for col in installments_columns if col != 'SK_ID_CURR'}
installments.rename(columns=new_installments_columns, inplace=True)

In [58]:
posh_cash = pd.read_csv('../data/dseb63_pos_cash_gb.csv', index_col=0)
posh_cash_columns = posh_cash.columns

new_posh_cash_columns = {col: 'POSH_CASH_' + col for col in posh_cash_columns if col != 'SK_ID_CURR'}
posh_cash.rename(columns=new_posh_cash_columns, inplace=True)

In [59]:
credit_card = pd.read_csv('../data/dseb63_credit_card_balance_gb.csv', index_col=0)
credit_card_columns = credit_card.columns

new_credit_card_columns = {col: 'CREDIT_CARD_' + col for col in credit_card_columns if col != 'SK_ID_CURR'}
credit_card.rename(columns=new_credit_card_columns, inplace=True)

In [60]:
prev_app = pd.read_csv('../data/prev_app_hanh.csv', index_col=0)
prev_app_columns = prev_app.columns

new_prev_app_columns = {col: 'PREV_APP_' + col for col in prev_app_columns if col != 'SK_ID_CURR'}
prev_app.rename(columns=new_prev_app_columns, inplace=True)

In [61]:
df = appl_train.merge(bureau, on='SK_ID_CURR', how='left')
df = df.merge(installments, on='SK_ID_CURR', how='left')
df = df.merge(posh_cash, on='SK_ID_CURR', how='left')
df = df.merge(credit_card, on='SK_ID_CURR', how='left')
df = df.merge(prev_app, on='SK_ID_CURR', how='left')

In [62]:
for col in df.columns:
    print(col, df[col].dtype)

TARGET int64
NAME_CONTRACT_TYPE object
CODE_GENDER object
FLAG_OWN_CAR object
FLAG_OWN_REALTY object
CNT_CHILDREN int64
AMT_INCOME_TOTAL float64
AMT_CREDIT float64
AMT_ANNUITY float64
AMT_GOODS_PRICE float64
NAME_TYPE_SUITE object
NAME_INCOME_TYPE object
NAME_EDUCATION_TYPE object
NAME_FAMILY_STATUS object
NAME_HOUSING_TYPE object
REGION_POPULATION_RELATIVE float64
DAYS_BIRTH int64
DAYS_EMPLOYED int64
DAYS_REGISTRATION float64
DAYS_ID_PUBLISH int64
OWN_CAR_AGE float64
FLAG_MOBIL int64
FLAG_EMP_PHONE int64
FLAG_WORK_PHONE int64
FLAG_CONT_MOBILE int64
FLAG_PHONE int64
FLAG_EMAIL int64
OCCUPATION_TYPE object
CNT_FAM_MEMBERS float64
REGION_RATING_CLIENT int64
REGION_RATING_CLIENT_W_CITY int64
WEEKDAY_APPR_PROCESS_START object
HOUR_APPR_PROCESS_START int64
REG_REGION_NOT_LIVE_REGION int64
REG_REGION_NOT_WORK_REGION int64
LIVE_REGION_NOT_WORK_REGION int64
REG_CITY_NOT_LIVE_CITY int64
REG_CITY_NOT_WORK_CITY int64
LIVE_CITY_NOT_WORK_CITY int64
ORGANIZATION_TYPE object
EXT_SOURCE_1 float64
EXT_

In [63]:
def RELU(series):
    return series.apply(lambda x: max(0, x))
    

In [64]:
def process_df(df):
    df = df.copy()
    
    df['EMERGENCYSTATE_MODE'].fillna('No', inplace=True)
    df['FONDKAPREMONT_MODE'].fillna('Unknown', inplace=True)
    df['WALLSMATERIAL_MODE'].fillna('Not Specified', inplace=True)
    df['OCCUPATION_TYPE'].replace('IT staff', 'High skill tech staff', inplace=True)
    df['OCCUPATION_TYPE'].replace('Realty agents', 'Sales staff', inplace=True)
    df['OCCUPATION_TYPE'].replace('HR staff', 'Laborers', inplace=True)
    df['OCCUPATION_TYPE'].fillna('Unknown', inplace=True)
    df['ORGANIZATION_TYPE'].replace('XNA', 'Unknown', inplace=True)
    
    map_week_day = {
    'MONDAY': 1,
    'TUESDAY': 2,
    'WEDNESDAY': 3,
    'THURSDAY': 4,
    'FRIDAY': 5,
    'SATURDAY': 6,
    'SUNDAY': 7
    }

    df['WEEKDAY_APPR_PROCESS_START'] = df['WEEKDAY_APPR_PROCESS_START'].map(map_week_day)

    map_edu = {
        'Lower secondary': 0,
        'Secondary / secondary special': 1,
        'Incomplete higher': 2,
        'Higher education': 5,
        'Academic degree': 10
    }

    df['NAME_EDUCATION_TYPE'].map(map_edu, na_action='ignore')

    df['NAME_FAMILY_STATUS'].replace('Unknown', 'Single / not married', inplace=True)
    df['CODE_GENDER'].replace('XNA', 'F', inplace=True)

    others = df['NAME_INCOME_TYPE'].value_counts().index[4:]
    df['NAME_INCOME_TYPE'].replace(others, 'Others', inplace=True)
    df['NAME_TYPE_SUITE'].fillna('Unaccompanied', inplace=True)

    df['OWN_CAR_AGE'].fillna(-100, inplace=True)
    # Steal code
    
    def group_organizations(org_type):
        if 'Trade' in org_type:
            return 'Trade'
        elif 'Industry' in org_type:
            return 'Industry'
        elif 'Business' in org_type:
            return 'Business Entity'
        elif 'Transport' in org_type:
            return 'Transport'
        else:
            return org_type
    
    df['ORGANIZATION_TYPE'] = df['ORGANIZATION_TYPE'].apply(group_organizations)
    
    med_income = df.groupby(['ORGANIZATION_TYPE', 'NAME_EDUCATION_TYPE'])['AMT_INCOME_TOTAL'].transform('median')
    med_income2 = df.groupby('ORGANIZATION_TYPE')['AMT_INCOME_TOTAL'].transform('median')
    df['income_ratio'] = df['AMT_INCOME_TOTAL'] / med_income
    df['income_ratio2'] = df['AMT_INCOME_TOTAL'] / med_income2
    df['true_annuity_div_income'] = df['AMT_ANNUITY'] / med_income
    df['true_annuity_div_income2'] = df['AMT_ANNUITY'] / med_income2
    # df['true_income_div_totalarea'] = med_income / df['TOTALAREA_MODE']
    # df['true_income_div_totalarea2'] = med_income2 / df['TOTALAREA_MODE']
    
    df['annuity_income_percentage'] = df['AMT_ANNUITY'] / df['AMT_INCOME_TOTAL']
    df['car_to_birth_ratio'] = RELU(df['OWN_CAR_AGE'] / df['DAYS_BIRTH'])
    df['car_to_employ_ratio'] = RELU(df['OWN_CAR_AGE'] / df['DAYS_EMPLOYED'])
    df['children_ratio'] = df['CNT_CHILDREN'] / df['CNT_FAM_MEMBERS']
    df['credit_to_annuity_ratio'] = df['AMT_CREDIT'] / df['AMT_ANNUITY']
    df['credit_to_goods_ratio'] = df['AMT_CREDIT'] / df['AMT_GOODS_PRICE']
    df['credit_to_income_ratio'] = df['AMT_CREDIT'] / df['AMT_INCOME_TOTAL']
    df['days_employed_percentage'] = df['DAYS_EMPLOYED'] / df['DAYS_BIRTH']
    df['ext_sources_mean'] = df[['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']].mean(axis=1)
    df['income_credit_percentage'] = df['AMT_INCOME_TOTAL'] / df['AMT_CREDIT']
    df['income_per_child'] = df['AMT_INCOME_TOTAL'] / (1 + df['CNT_CHILDREN'])
    df['income_per_person'] = df['AMT_INCOME_TOTAL'] / df['CNT_FAM_MEMBERS']
    df['payment_rate'] = df['AMT_ANNUITY'] / df['AMT_CREDIT']
    df['phone_to_birth_ratio'] = RELU(df['DAYS_LAST_PHONE_CHANGE'] / df['DAYS_BIRTH'])
    df['phone_to_employ_ratio'] = RELU(df['DAYS_LAST_PHONE_CHANGE'] / df['DAYS_EMPLOYED'])
    df['external_sources_weighted'] = df.EXT_SOURCE_1 * 2 + df.EXT_SOURCE_2 * 3 + df.EXT_SOURCE_3 * 4

    df['sin_HOUR_APPR_PROCESS_START'] = np.sin(2 * np.pi * df['HOUR_APPR_PROCESS_START'] / 24)
    df['cos_HOUR_APPR_PROCESS_START'] = np.cos(2 * np.pi * df['HOUR_APPR_PROCESS_START'] / 24)
    df.drop(columns=['HOUR_APPR_PROCESS_START'], inplace=True)

    # numerical transformation
    df['DAYS_EMPLOYED'].replace(365243,0, inplace=True)
    df['REGION_POPULATION_RELATIVE'] = np.sqrt(df['REGION_POPULATION_RELATIVE'])
    df['APARTMENTS_AVG'] = np.log1p(50 * df['APARTMENTS_AVG'])
    df['YEARS_BUILD_AVG'] = df['YEARS_BUILD_AVG'] ** 3
    df['COMMONAREA_AVG'] = df['COMMONAREA_AVG'] ** (1/2)
    df['ELEVATORS_AVG'] = df['ELEVATORS_AVG'] ** (1/10)
    df['ENTRANCES_AVG'] = df['ENTRANCES_AVG'] ** (1/3)
    df['FLOORSMAX_AVG'] = df['FLOORSMAX_AVG'] ** (1/2.5)
    df['FLOORSMIN_AVG'] = df['FLOORSMIN_AVG'] ** (1/2.2)
    df['LANDAREA_AVG'] = df['LANDAREA_AVG'] ** (1/5)
    df['LIVINGAPARTMENTS_AVG'] = df['LIVINGAPARTMENTS_AVG'] ** (1/3)
    df['LIVINGAREA_AVG'] = df['LIVINGAREA_AVG'] ** (1/2)
    df['NONLIVINGAPARTMENTS_AVG'] = df['NONLIVINGAPARTMENTS_AVG'] ** (1/5)
    df['NONLIVINGAREA_AVG'] = df['NONLIVINGAREA_AVG'] ** (1/3)
    df['OBS_30_CNT_SOCIAL_CIRCLE'] = df['OBS_30_CNT_SOCIAL_CIRCLE'] ** (1/4)
    df['DEF_30_CNT_SOCIAL_CIRCLE'] = df['DEF_30_CNT_SOCIAL_CIRCLE'] ** (1/4)
    df['OBS_60_CNT_SOCIAL_CIRCLE'] = df['OBS_60_CNT_SOCIAL_CIRCLE'] ** (1/4)
    df['DEF_60_CNT_SOCIAL_CIRCLE'] = df['DEF_60_CNT_SOCIAL_CIRCLE'] ** (1/4)
    
    return df
df = process_df(df)

/tmp/ipykernel_299255/2574229585.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['EMERGENCYSTATE_MODE'].fillna('No', inplace=True)
/tmp/ipykernel_299255/2574229585.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True

In [65]:
def display_missing_data_info(dataframe, ascending=False):
    missing_values_count = dataframe.isnull().sum()
    missing_values_percentage = (dataframe.isnull().mean() * 100)
    missing_data = pd.DataFrame({
        'Missing Values': missing_values_count,
        'Percentage (%)': missing_values_percentage
    })
    
    missing_data = missing_data[missing_data['Missing Values'] > 0]
    
    sorted_missing_data = missing_data.sort_values(by='Missing Values', ascending=ascending)

    print(sorted_missing_data)
    return sorted_missing_data

In [66]:
display_missing_data_info(df)

                               Missing Values  Percentage (%)
BUREAU_BAD_DEBT_SUM_CREDIT             245993       99.993496
BUREAU_BAD_DEBT_DURATION               245993       99.993496
BUREAU_COUNT_BAD_DEBT                  245993       99.993496
BUREAU_BAD_DEBT_LAST_365_DAYS          245993       99.993496
BUREAU_BAD_DEBT_LAST_730_DAYS          245993       99.993496
...                                       ...             ...
true_annuity_div_income                    10        0.004065
children_ratio                              1        0.000406
income_per_person                           1        0.000406
DAYS_LAST_PHONE_CHANGE                      1        0.000406
CNT_FAM_MEMBERS                             1        0.000406

[753 rows x 2 columns]


,Missing Values,Percentage (%)
BUREAU_BAD_DEBT_SUM_CREDIT,245993,99.993496
BUREAU_BAD_DEBT_DURATION,245993,99.993496
BUREAU_COUNT_BAD_DEBT,245993,99.993496
BUREAU_BAD_DEBT_LAST_365_DAYS,245993,99.993496
BUREAU_BAD_DEBT_LAST_730_DAYS,245993,99.993496
...,...,...
true_annuity_div_income,10,0.004065
children_ratio,1,0.000406
income_per_person,1,0.000406
DAYS_LAST_PHONE_CHANGE,1,0.000406


In [67]:
from sklearn.preprocessing import OneHotEncoder

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PowerTransformer, OrdinalEncoder, LabelEncoder
from sklearn.impute import SimpleImputer

In [68]:
df_train = df.copy()
df_train.dropna(subset=['TARGET'], inplace=True)

In [69]:
df_train['TARGET'].value_counts()

TARGET
0    226133
1     19876
Name: count, dtype: int64

In [70]:
X = df_train.drop(columns=['TARGET', 'SK_ID_CURR'])
y = df_train['TARGET']

In [71]:
for col in df.columns:
    if df[col].dtype == 'object':
        print(col)

NAME_CONTRACT_TYPE
CODE_GENDER
FLAG_OWN_CAR
FLAG_OWN_REALTY
NAME_TYPE_SUITE
NAME_INCOME_TYPE
NAME_EDUCATION_TYPE
NAME_FAMILY_STATUS
NAME_HOUSING_TYPE
OCCUPATION_TYPE
ORGANIZATION_TYPE
FONDKAPREMONT_MODE
HOUSETYPE_MODE
WALLSMATERIAL_MODE
EMERGENCYSTATE_MODE


In [72]:
ordinal_cols = ['FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'EMERGENCYSTATE_MODE', 'CODE_GENDER', 'NAME_CONTRACT_TYPE']
float_cols = []
int_cols = []
cate_cols = []
flag_cols = []
for col in X.columns:
    if col not in ordinal_cols:
        if X[col].dtype == 'float64':
            float_cols.append(col)
        elif X[col].dtype == 'int64':
            if 'FLAG' in col:
                flag_cols.append(col)
            else:
                int_cols.append(col)
        else:
            cate_cols.append(col)


In [73]:
df_train[float_cols+int_cols] = df_train[float_cols+int_cols].clip(-999999999, 999999999)

In [74]:
len(int_cols)

13

In [75]:
df_train.describe()

,TARGET,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,DAYS_REGISTRATION,...,ext_sources_mean,income_credit_percentage,income_per_child,income_per_person,payment_rate,phone_to_birth_ratio,phone_to_employ_ratio,external_sources_weighted,sin_HOUR_APPR_PROCESS_START,cos_HOUR_APPR_PROCESS_START
count,246009.000000,246009.000000,2.460090e+05,2.460090e+05,245999.000000,2.457850e+05,246009.000000,246009.000000,246009.000000,246009.000000,...,245875.000000,246009.000000,2.460090e+05,2.460080e+05,245999.000000,246009.000000,246009.000000,87835.000000,2.460090e+05,246009.000000
mean,0.080794,0.416229,1.684589e+05,5.996410e+05,27120.238129,5.389827e+05,0.136951,-16038.946343,-1954.048015,-4985.222248,...,0.509317,0.398896,1.393917e+05,9.295885e+04,0.053680,0.063071,0.897447,4.593643,-1.199816e-02,-0.690230
std,0.272519,0.720664,1.045297e+05,4.030980e+05,14510.634712,3.700520e+05,0.045819,4361.265483,2307.975179,3521.391396,...,0.149795,0.340379,1.011083e+05,7.315516e+04,0.022481,0.055954,4.593887,1.182048,6.282103e-01,0.358877
min,0.000000,0.000000,2.565000e+04,4.500000e+04,1615.500000,4.050000e+04,0.017029,-25229.000000,-17912.000000,-23416.000000,...,0.000011,0.011801,3.000000e+03,2.812500e+03,0.022073,0.000000,0.000000,0.224072,-1.000000e+00,-1.000000
25%,0.000000,0.000000,1.125000e+05,2.700000e+05,16506.000000,2.385000e+05,0.100030,-19682.000000,-2758.000000,-7482.000000,...,0.413514,0.193603,7.875000e+04,4.725000e+04,0.036867,0.017029,0.000000,3.797276,-5.000000e-01,-0.965926
50%,0.000000,0.000000,1.462500e+05,5.147775e+05,24903.000000,4.500000e+05,0.137295,-15755.000000,-1212.000000,-4504.000000,...,0.524666,0.306272,1.170000e+05,7.500000e+04,0.050000,0.050534,0.294048,4.684987,1.224647e-16,-0.866025
75%,0.000000,1.000000,2.025000e+05,8.086500e+05,34654.500000,6.795000e+05,0.169302,-12418.000000,-288.000000,-2006.000000,...,0.622943,0.495376,1.800000e+05,1.125000e+05,0.064045,0.098070,0.785386,5.476326,5.000000e-01,-0.500000
max,1.000000,19.000000,1.350000e+07,4.050000e+06,258025.500000,4.050000e+06,0.269273,-7489.000000,0.000000,0.000000,...,0.878903,15.000000,1.350000e+07,6.750000e+06,0.124430,0.361365,1784.000000,7.677298,1.000000e+00,1.000000


In [76]:
description = df_train.describe()

In [77]:
description.columns

Index(['TARGET', 'CNT_CHILDREN', 'AMT_INCOME_TOTAL', 'AMT_CREDIT',
       'AMT_ANNUITY', 'AMT_GOODS_PRICE', 'REGION_POPULATION_RELATIVE',
       'DAYS_BIRTH', 'DAYS_EMPLOYED', 'DAYS_REGISTRATION',
       ...
       'ext_sources_mean', 'income_credit_percentage', 'income_per_child',
       'income_per_person', 'payment_rate', 'phone_to_birth_ratio',
       'phone_to_employ_ratio', 'external_sources_weighted',
       'sin_HOUR_APPR_PROCESS_START', 'cos_HOUR_APPR_PROCESS_START'],
      dtype='object', length=810)

In [78]:
description.columns[description.values[-1,:] > 999999998]

Index(['BUREAU_GENERAL_AMT_CREDIT_SUM_sum', 'BUREAU_SUM_CONSUMER_CREDIT',
       'BUREAU_CLOSE_AMT_CREDIT_SUM'],
      dtype='object')

In [79]:
keep_flag = flag_cols.copy()
# keep_flag = []
# for col in flag_cols:
#     if 'FLAG_DOCUMENT' not in col:
#         keep_flag.append(col)

In [80]:
for col in float_cols:
    X[col].fillna(0, inplace=True)
    
for col in int_cols:
    X[col].fillna(0, inplace=True)
for col in cate_cols:
    X[col].fillna('Unknown', inplace=True)

/tmp/ipykernel_299255/1735439884.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  X[col].fillna(0, inplace=True)
/tmp/ipykernel_299255/1735439884.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.meth

In [81]:
# Fill missing values in categorical columns


In [82]:
from sklearn.model_selection import train_test_split

X_train_, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.1, random_state=42, stratify=y
)

In [83]:
y_train = y_train.to_numpy()
y_val = y_val.to_numpy()

In [84]:
onehot_transformer = OneHotEncoder(handle_unknown='ignore')
power_transformer = PowerTransformer()
ordinal_transformer = OrdinalEncoder()
scaler_transformer = StandardScaler()

preprocessor = ColumnTransformer(
    transformers=[
        ('onehot', onehot_transformer, cate_cols),
        ('power', power_transformer, float_cols), # power_transformer
        ('ordinal', ordinal_transformer, ordinal_cols),
        ('scale', scaler_transformer, int_cols),
        ('flag', 'passthrough', keep_flag)
        
    ]
)


In [85]:
X.head()

,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,NAME_TYPE_SUITE,...,ext_sources_mean,income_credit_percentage,income_per_child,income_per_person,payment_rate,phone_to_birth_ratio,phone_to_employ_ratio,external_sources_weighted,sin_HOUR_APPR_PROCESS_START,cos_HOUR_APPR_PROCESS_START
0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,1129500.0,Family,...,0.466757,0.208736,270000.0,135000.0,0.027598,0.049389,0.696970,0.0,0.258819,-0.965926
1,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,297000.0,Unaccompanied,...,0.650442,0.431748,135000.0,67500.0,0.094941,0.032465,0.203027,0.0,-0.965926,-0.258819
2,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,513000.0,Unaccompanied,...,0.322738,0.236842,121500.0,121500.0,0.042623,0.055489,0.364055,0.0,0.258819,-0.965926
3,Cash loans,M,N,Y,0,99000.0,490495.5,27517.5,454500.0,"Spouse, partner",...,0.487726,0.201837,99000.0,49500.0,0.056101,0.149696,1.596977,0.0,-0.866025,-0.500000
4,Cash loans,M,Y,Y,0,360000.0,1530000.0,42075.0,1530000.0,Unaccompanied,...,0.627467,0.235294,360000.0,180000.0,0.027500,0.056764,2.383073,0.0,-0.866025,-0.500000


In [86]:
X_val.shape

(24601, 823)

In [87]:
X_train = preprocessor.fit_transform(X_train_)
X_train

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 1., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [88]:
X_val = preprocessor.transform(X_val)

In [89]:
X_train.shape   

(221408, 909)

In [90]:
cat_feature = preprocessor.named_transformers_['onehot'].get_feature_names_out(cate_cols)
cat_feature

feature_names = np.concatenate([cat_feature, float_cols, ordinal_cols, int_cols, keep_flag])

In [91]:
!pip install imbalanced-learn

## Upsample - Downsample

In [92]:
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import ClusterCentroids, TomekLinks
from sklearn.model_selection import train_test_split

In [93]:
# smote = SMOTE(sampling_strategy=0.2, random_state=42, n_jobs=-1)
# X_resampled, y_resampled = smote.fit_resample(X_train, y_train)

In [94]:
# undersampler = TomekLinks(n_jobs=-1)
# X_resampled, y_resampled = smote.fit_resample(X_train, y_train)

### Manual double

In [95]:
from sklearn.utils import resample


In [96]:
y_train

array([0, 0, 0, ..., 0, 0, 0])

In [97]:
def resample_data(X_train, y_train, rate = 0.2):
    posidx = (y_train == 1)
    negidx = (y_train == 0)

    y_train2 = y_train.reshape(-1, 1)

    train_data = np.concatenate([X_train, y_train2], axis=1)

    train_data_pos = train_data[posidx]
    train_data_neg = train_data[negidx]

    train_data_resampled = resample(train_data_pos, n_samples=int(len(train_data_neg) * rate), random_state=42)

    train_data_resampled = np.concatenate([train_data_resampled, train_data_neg])

    np.random.shuffle(train_data_resampled)
    X_resampled = train_data_resampled[:, :-1]
    y_resampled = train_data_resampled[:, -1]
    
    return X_resampled, y_resampled

In [98]:
X_resampled, y_resampled = resample_data(X_train, y_train, rate=0.4)

In [99]:
# X_resampled = X_train
# y_resampled = y_train

In [100]:

from sklearn.model_selection import cross_val_score, StratifiedKFold, cross_validate
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

In [101]:
from sklearn.decomposition import PCA
import pandas as pd
import numpy as np

In [102]:
pca = PCA()
pca.fit(X_train)

PCA()

In [103]:
explained_variance = pca.explained_variance_ratio_

# Get the absolute values of PCA components (loadings)
feature_contributions = np.abs(pca.components_)

# Aggregate feature importance scores
# Weight each feature's contribution by the variance explained by the component
weighted_contributions = feature_contributions * explained_variance[:, np.newaxis]

# Sum weighted contributions across components
feature_importance = weighted_contributions.sum(axis=0)

# Create a DataFrame for ranking
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': feature_importance
}).sort_values(by='Importance', ascending=False)
importance_df

,Feature,Importance
649,PREV_APP_PREV_CHANNEL_TYPE_Credit and cash off...,3.004852e-02
499,PREV_APP_PREV_APP_CREDIT_PERC_MIN,2.994485e-02
707,PREV_APP_APPROVED_APP_CREDIT_PERC_MIN,2.975137e-02
539,PREV_APP_PREV_FLAG_LAST_APPL_PER_CONTRACT_Y_MEAN,2.961483e-02
570,PREV_APP_PREV_NAME_CONTRACT_STATUS_Canceled_SUM,2.936595e-02
...,...,...
786,PREV_APP_APPROVED_NAME_CONTRACT_STATUS_Cancele...,3.419399e-21
787,PREV_APP_APPROVED_NAME_CONTRACT_STATUS_Refused...,3.266189e-21
791,PREV_APP_APPROVED_NAME_CONTRACT_STATUS_nan_MEAN,3.065029e-21
792,PREV_APP_APPROVED_NAME_CONTRACT_STATUS_nan_SUM,2.465882e-21


In [104]:
raise Exception('Stop here')

Exception: Stop here

In [ ]:
def select_features_with_pca(X, threshold=0.1):
    """
    Select features contributing to principal components above a certain threshold.

    Parameters:
    X (numpy array or pandas DataFrame): Input feature matrix.
    threshold (float): Contribution threshold for selecting features (default is 0.1).

    Returns:
    selected_columns (list): List of selected feature names or indices.
    selected_data (DataFrame): Data with selected features.
    """
    # Convert to DataFrame if X is a NumPy array
    # if isinstance(X, np.ndarray):
    #     X = pd.DataFrame(X, columns=[f"Feature{i+1}" for i in range(X.shape[1])])
    
    # Apply PCA
    pca = PCA()
    pca.fit(X)
    
    # Calculate absolute loadings (contributions of features to components)
    explained_variance = pca.explained_variance_ratio_
    feature_contributions = np.abs(pca.components_)
    weighted_contributions = feature_contributions * explained_variance[:, np.newaxis]
    # Identify features exceeding the threshold for any component
    
    feature_mask = (weighted_contributions > threshold).any(axis=0)
    
    # Filter the data to keep only selected features
    # selected_data = X[feature_mask, :]
    
    return feature_mask

IF USING PCA

In [ ]:
feature_map = select_features_with_pca(X_train, threshold=0.00001)

In [ ]:
# feature_map = np.ones(len(feature_names), dtype=bool)

In [ ]:
feature_map.sum()

809

In [ ]:
# X_df = pd.DataFrame(X_train, columns=feature_names)
# X_train_selected_columns, X_train_selected = select_features_with_pca(X_df, threshold=0.0001)

In [ ]:
logistic_model = LogisticRegression(random_state=42, max_iter=10000, n_jobs=-1, class_weight='balanced')

In [ ]:
dt_model = DecisionTreeClassifier(random_state=42, class_weight = 'balanced', min_samples_leaf=2, criterion='log_loss')

In [ ]:
stacking_model = StackingClassifier(
    estimators=[
        ('lr', logistic_model),
        ('dt', dt_model)
    ],
    final_estimator=LogisticRegression(class_weight='balanced', random_state=42, max_iter=10000),  # Final model for stacking
    cv=5
)

In [ ]:
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:
from sklearn.metrics import make_scorer, roc_auc_score
def gini_coefficient(y_true, y_pred):
    """
    Calculate the Gini coefficient using predictions and true labels.
    
    Parameters:
    y_true (array-like): True binary labels.
    y_pred (array-like): Predicted probabilities.
    
    Returns:
    float: Gini coefficient.
    """
    auc = roc_auc_score(y_true, y_pred)  # AUC calculation
    return 2 * auc - 1  # Gini coefficient

In [ ]:
gini_scorer = make_scorer(gini_coefficient, needs_proba=True)

/home/quanghung20gg/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_scorer.py:548: FutureWarning: The `needs_threshold` and `needs_proba` parameter are deprecated in version 1.4 and will be removed in 1.6. You can either let `response_method` be `None` or set it to `predict` to preserve the same behaviour.
  warnings.warn(


In [ ]:
print("Logistic Regression Cross-Validation Scores:")
results = cross_validate(logistic_model, X_resampled[:, feature_map], y_resampled, cv=kfold, scoring=gini_scorer, n_jobs=-1,return_estimator=True)

for i, estimator in enumerate(results['estimator']):
    y_pred = estimator.predict_proba(X_val[:, feature_map])
    gini_scores = gini_coefficient(y_val, y_pred[:,1])
    print(f"- Gini Score for fold {i}: {gini_scores:.4f}")

Logistic Regression Cross-Validation Scores:
- Gini Score for fold 0: 0.5437
- Gini Score for fold 1: 0.5415
- Gini Score for fold 2: 0.5425
- Gini Score for fold 3: 0.5426
- Gini Score for fold 4: 0.5424


### Number transformation
Logistic Regression Cross-Validation Scores:
- Gini Score for fold 0: 0.5364
- Gini Score for fold 1: 0.5359
- Gini Score for fold 2: 0.5350
- Gini Score for fold 3: 0.5366
- Gini Score for fold 4: 0.5334



### RAW
Logistic Regression Cross-Validation Scores:
- Gini Score for fold 0: 0.5146
- Gini Score for fold 1: 0.5154
- Gini Score for fold 2: 0.5152
- Gini Score for fold 3: 0.5153

### UPSAMPLE RAW

Logistic Regression Cross-Validation Scores:
- Gini Score for fold 0: 0.5135
- Gini Score for fold 1: 0.5167
- Gini Score for fold 2: 0.5118
- Gini Score for fold 3: 0.5127

### PCA
Logistic Regression Cross-Validation Scores:
- Gini Score for fold 0: 0.5337
- Gini Score for fold 1: 0.5343
- Gini Score for fold 2: 0.5326
- Gini Score for fold 3: 0.5351



### Drop Flag

Logistic Regression Cross-Validation Scores:
- Gini Score for fold 0: 0.5129
- Gini Score for fold 1: 0.5156
- Gini Score for fold 2: 0.5125
- Gini Score for fold 3: 0.5151

### Super upsample + PCA
- Gini Score for fold 0: 0.5156
- Gini Score for fold 1: 0.5158
- Gini Score for fold 2: 0.5183
- Gini Score for fold 3: 0.5186


### ADD CREDIT CARD
Logistic Regression Cross-Validation Scores:
- Gini Score for fold 0: 0.5220
- Gini Score for fold 1: 0.5207
- Gini Score for fold 2: 0.5230
- Gini Score for fold 3: 0.5233

### Add app prev
Logistic Regression Cross-Validation Scores:
- Gini Score for fold 0: 0.5349
- Gini Score for fold 1: 0.5357
- Gini Score for fold 2: 0.5352
- Gini Score for fold 3: 0.5347


In [ ]:
# print("Decision Tree Cross-Validation Scores:")
# results = cross_validate(dt_model, X_resampled[:, feature_map], y_resampled, cv=kfold, scoring=gini_scorer, n_jobs=-1,return_estimator=True)

# for i, estimator in enumerate(results['estimator']):
#     y_pred = estimator.predict_proba(X_val[:, feature_map])
#     gini_scores = gini_coefficient(y_val, y_pred[:,1])
#     print(f"Gini Score for fold {i}: {gini_scores:.4f}")

In [ ]:
# print("Stacking Scores:")
# stacking_model.fit(X_resampled[:, feature_map], y_resampled)
# y_pred = stacking_model.predict_proba(X_val[:, feature_map])
# gini_scores = gini_coefficient(y_val, y_pred[:,1])
# print(f"Gini Scores: {gini_scores}")

Stacking Cross-Validation Scores:

Gini Scores for each fold: [0.45493243 0.47045351 0.46576538 0.47327105 0.45963799]

Mean Gini Score: 0.4648

In [ ]:
df_test = pd.read_csv('../data/dseb63_application_test.csv',  index_col=0)


In [ ]:
df_test = df_test.merge(bureau, on='SK_ID_CURR', how='left')
df_test = df_test.merge(installments, on='SK_ID_CURR', how='left')
df_test = df_test.merge(posh_cash, on='SK_ID_CURR', how='left')
df_test = df_test.merge(credit_card, on='SK_ID_CURR', how='left')
df_test = df_test.merge(prev_app, on='SK_ID_CURR', how='left')

In [ ]:
df_test = process_df(df_test)

/tmp/ipykernel_205842/1357583645.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['EMERGENCYSTATE_MODE'].fillna('No', inplace=True)
/tmp/ipykernel_205842/1357583645.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True

In [ ]:
df_test 

,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,NAME_TYPE_SUITE,...,days_employed_percentage,ext_sources_mean,income_credit_percentage,income_per_child,income_per_person,payment_rate,phone_to_birth_ratio,phone_to_employ_ratio,sin_HOUR_APPR_PROCESS_START,cos_HOUR_APPR_PROCESS_START
0,Cash loans,M,Y,N,2,207000.0,465457.5,52641.0,418500.0,Unaccompanied,...,0.057306,0.427100,0.444724,69000.0,51750.0,0.113095,0.000150,0.002625,0.258819,-0.965926
1,Cash loans,F,Y,Y,0,247500.0,1281712.5,48946.5,1179000.0,Unaccompanied,...,0.077209,0.522778,0.193101,247500.0,247500.0,0.038188,0.072473,0.938650,0.500000,-0.866025
2,Cash loans,F,Y,N,0,202500.0,495000.0,39109.5,495000.0,Unaccompanied,...,0.035684,0.422321,0.409091,202500.0,101250.0,0.079009,0.080136,2.245696,-0.866025,-0.500000
3,Cash loans,F,N,Y,0,247500.0,254700.0,24939.0,225000.0,Unaccompanied,...,0.355753,0.653968,0.971731,247500.0,247500.0,0.097915,0.101906,0.286451,-0.500000,-0.866025
4,Cash loans,M,N,Y,0,112500.0,308133.0,15862.5,234000.0,Unaccompanied,...,0.054361,0.617316,0.365102,112500.0,112500.0,0.051479,0.008511,0.156561,0.258819,-0.965926
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61497,Cash loans,M,Y,Y,0,171000.0,521280.0,23089.5,450000.0,Family,...,-15.350859,0.514059,0.328039,171000.0,85500.0,0.044294,0.067793,0.000000,0.500000,-0.866025
61498,Revolving loans,M,Y,Y,2,450000.0,900000.0,45000.0,900000.0,Family,...,0.112609,0.430288,0.500000,150000.0,112500.0,0.050000,0.087960,0.781116,-0.258819,-0.965926
61499,Cash loans,F,N,Y,0,225000.0,202500.0,24030.0,202500.0,Family,...,0.015172,0.341720,1.111111,225000.0,112500.0,0.118667,0.151075,9.957219,-0.258819,-0.965926
61500,Cash loans,M,N,Y,0,121500.0,254700.0,30357.0,225000.0,Family,...,-15.457404,0.667064,0.477032,121500.0,60750.0,0.119187,0.067163,0.000000,0.866025,0.500000


In [ ]:
for col in float_cols:
    df_test[col].fillna(0, inplace=True)
    
for col in int_cols:
    df_test[col].fillna(0, inplace=True)
for col in cate_cols:
    df_test[col].fillna('Unknown', inplace=True)

/tmp/ipykernel_205842/1295519593.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_test[col].fillna(0, inplace=True)


/tmp/ipykernel_205842/1295519593.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_test[col].fillna(0, inplace=True)
/tmp/ipykernel_205842/1295519593.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'd

In [ ]:
X_test = df_test.drop(columns=['SK_ID_CURR'])

In [ ]:
X_test[float_cols+int_cols] = X_test[float_cols+int_cols].clip(-999999999, 999999999)

In [ ]:
X_test

,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,NAME_TYPE_SUITE,...,days_employed_percentage,ext_sources_mean,income_credit_percentage,income_per_child,income_per_person,payment_rate,phone_to_birth_ratio,phone_to_employ_ratio,sin_HOUR_APPR_PROCESS_START,cos_HOUR_APPR_PROCESS_START
0,Cash loans,M,Y,N,2,207000.0,465457.5,52641.0,418500.0,Unaccompanied,...,0.057306,0.427100,0.444724,69000.0,51750.0,0.113095,0.000150,0.002625,0.258819,-0.965926
1,Cash loans,F,Y,Y,0,247500.0,1281712.5,48946.5,1179000.0,Unaccompanied,...,0.077209,0.522778,0.193101,247500.0,247500.0,0.038188,0.072473,0.938650,0.500000,-0.866025
2,Cash loans,F,Y,N,0,202500.0,495000.0,39109.5,495000.0,Unaccompanied,...,0.035684,0.422321,0.409091,202500.0,101250.0,0.079009,0.080136,2.245696,-0.866025,-0.500000
3,Cash loans,F,N,Y,0,247500.0,254700.0,24939.0,225000.0,Unaccompanied,...,0.355753,0.653968,0.971731,247500.0,247500.0,0.097915,0.101906,0.286451,-0.500000,-0.866025
4,Cash loans,M,N,Y,0,112500.0,308133.0,15862.5,234000.0,Unaccompanied,...,0.054361,0.617316,0.365102,112500.0,112500.0,0.051479,0.008511,0.156561,0.258819,-0.965926
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61497,Cash loans,M,Y,Y,0,171000.0,521280.0,23089.5,450000.0,Family,...,-15.350859,0.514059,0.328039,171000.0,85500.0,0.044294,0.067793,0.000000,0.500000,-0.866025
61498,Revolving loans,M,Y,Y,2,450000.0,900000.0,45000.0,900000.0,Family,...,0.112609,0.430288,0.500000,150000.0,112500.0,0.050000,0.087960,0.781116,-0.258819,-0.965926
61499,Cash loans,F,N,Y,0,225000.0,202500.0,24030.0,202500.0,Family,...,0.015172,0.341720,1.111111,225000.0,112500.0,0.118667,0.151075,9.957219,-0.258819,-0.965926
61500,Cash loans,M,N,Y,0,121500.0,254700.0,30357.0,225000.0,Family,...,-15.457404,0.667064,0.477032,121500.0,60750.0,0.119187,0.067163,0.000000,0.866025,0.500000


In [ ]:
np.where(X_test.describe().values[-1, :]>1000000000000)

(array([], dtype=int64),)

In [ ]:
X_test.describe().columns[[370, 371, 584, 585]]

Index(['CREDIT_CARD_36_MONTH_UTILIZATION_sum',
       'CREDIT_CARD_36_MONTH_FLAG_UTILIZATION_LESS_50_sum',
       'PREV_APP_PREV_NAME_SELLER_INDUSTRY_Consumer electronics_MEAN',
       'PREV_APP_PREV_NAME_SELLER_INDUSTRY_Furniture_MEAN'],
      dtype='object')

In [ ]:
X_test = preprocessor.transform(X_test)


In [ ]:
X_val_resampled, y_val_resampled = resample_data(X_val, y_val, rate=0.2)

In [ ]:
X_all = np.concatenate([X_resampled, X_val_resampled])
y_all = np.concatenate([y_resampled, y_val_resampled])

In [ ]:
logistic_model.fit(X_all[:, feature_map], y_all)

LogisticRegression(class_weight='balanced', max_iter=10000, n_jobs=-1,
                   random_state=42)

In [ ]:
y_pred = logistic_model.predict_proba(X_test[:, feature_map])   

In [ ]:
# stacking_model.fit(X_all[:, feature_map], y_all)

In [ ]:
# y_pred = stacking_model.predict_proba(X_test[:, feature_map])

In [ ]:
# gini_coefficient(y, y_pred[:, 1])

In [ ]:
submission = pd.DataFrame({
    'SK_ID_CURR': df_test['SK_ID_CURR'],
    'TARGET': y_pred[:, 1]
})
submission.to_csv('../submission/dseb63_log_super_resample_super_feat.csv', index=False)